# Notebook 02 — Energy layer: solar and wind resource, irrigation demand and system sizing at four water-scarce sites

**Purpose.** Build the energy side of the digital twin and size an off-grid PV–wind–battery–pump–tank farm microgrid for each site before any controller is compared. The microgrid serves (i) irrigation pumping, which is seasonal and flexible through the tank, and (ii) year-round farm loads (farmhouse appliances, a produce cold room, a workshop), which are critical and partly at night.

**Sites.** Bushland (Texas, USA), Jodhpur (Rajasthan, India), Taroudant (Souss-Massa, Morocco), Azraq (Jordan).

**Data (public, downloaded here).** NASA POWER hourly (2001–2024: GHI, DNI, DHI, temperature, humidity, rain, wind at 10 and 50 m, pressure) and daily (agro-climatology community) for FAO-56 ET0.

**Models.** PV: NOAA solar position, isotropic-sky plane-of-array irradiance (tilt = latitude, equator-facing), NOCT cell temperature, PVWatts-type output with 14% system losses and 96% inverter efficiency; cross-checked against pvlib. Wind: hourly shear exponent from the 10 m and 50 m winds, power-law extrapolation to a 30 m hub, IEC air-density correction, generic small-turbine curve (3/12/25 m/s), 10% losses. Demand for sizing: FAO-56 single-crop-coefficient maize (5 ha, drip efficiency 0.90), simple daily effective rain. Farm load: appliance-based stochastic profile (RAMP-style; randomised daily use windows, seeded) with temperature-dependent fans and cold-room compressor. System: variable-speed pump (wire-to-water efficiency 0.55, 30% minimum power), tank, Li-ion battery (SoC 10–90%, 0.96/0.96 charge/discharge, 0.5C), rule-based dispatch (farm load first, then pump, then battery; the battery tops up the pump only above a 50% night reserve).

**Sizing.** Grid search over PV, wind, battery and tank sizes; feasible if the water loss-of-supply probability (WLPSP) is at most 5% and the farm-load energy loss-of-supply probability (ELPSP) at most 2% on 2001–2016; least net present cost (NPC) chosen; reliability then re-checked on 2020–2024 (not used for sizing). Diesel reference: a genset supplying the same pumping and farm-load energy at constant efficiency (favours diesel). Costs are sourced base values (IRENA 2024, PNNL distributed wind 2025/2026, NREL ATB 2024b; tank and pump assumed) with site-specific 2024 diesel prices; sources in `energy.COST_SOURCES`, bounds in `energy.COST_SCENARIOS`.

**Runtime.** CPU; about 45–75 min (sizing about 4–6 min per site, head sensitivity about 15 min, hourly data download about 10–15 min on the first run). Keep the browser tab open. **Output:** `export_nb02.zip`.

In [ ]:
!pip -q install pvlib pyarrow
import os, sys, json, gzip, time, platform, shutil, urllib.request, urllib.parse
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive, files
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/solar_wind_irrigation'
RAWP = f'{BASE}/data/raw/nasa_power'; PROC = f'{BASE}/data/processed'; OUT = f'{BASE}/outputs/nb02'; SRC = f'{BASE}/src'
for d in (RAWP, PROC, OUT, SRC): os.makedirs(d, exist_ok=True)
sys.path.insert(0, SRC)

## 1. Shared modules and site list (identical to the locally tested `src/energy.py`, `src/et0.py`, `src/sites.json`)

In [ ]:
%%writefile {SRC}/energy.py
"""Energy layer of the digital twin: PV and wind generation from NASA POWER hourly data, FAO-56 irrigation demand for
sizing, an hourly PV-wind-battery-pump-tank simulation with a rule-based dispatch, and simple techno-economics.

All functions are plain numpy/pandas so they can be tested without pvlib; the Colab notebook cross-checks the PV model
against pvlib. Units: kW, kWh, m3, mm, deg C, m/s. Times are UTC unless stated.
"""
import gzip
import json
from pathlib import Path

import numpy as np
import pandas as pd

RHO_W, G = 1000.0, 9.81


# ---------------------------------------------------------------- data
def load_power_hourly(folder, site):
    """Concatenate NASA POWER hourly files <site>_hourly_<year>.json.gz into one UTC-indexed DataFrame."""
    frames = []
    for f in sorted(Path(folder).glob(f"{site}_hourly_*.json.gz")):
        p = json.load(gzip.open(f))["properties"]["parameter"]
        d = pd.DataFrame(p)
        d.index = pd.to_datetime(d.index, format="%Y%m%d%H", utc=True)
        frames.append(d)
    df = pd.concat(frames).sort_index()
    df = df[~df.index.duplicated()]
    return df.replace(-999.0, np.nan)


def load_power_daily(path):
    """Daily POWER file -> (DataFrame, elevation of the POWER grid cell in m, from the response geometry)."""
    j = json.load(gzip.open(path))
    d = pd.DataFrame(j["properties"]["parameter"])
    d.index = pd.to_datetime(d.index, format="%Y%m%d")
    coords = j.get("geometry", {}).get("coordinates", [None, None, None])
    return d.replace(-999.0, np.nan), (coords[2] if len(coords) > 2 else None)


# ---------------------------------------------------------------- solar geometry (NOAA general solar position)
def solar_position(times_utc, lat, lon):
    """Zenith and azimuth (deg, azimuth clockwise from north) for UTC timestamps (NOAA spreadsheet equations)."""
    t = pd.DatetimeIndex(times_utc)
    doy = t.dayofyear.values
    hr = t.hour.values + t.minute.values / 60.0
    gam = 2 * np.pi / 365 * (doy - 1 + (hr - 12) / 24)
    eot = 229.18 * (0.000075 + 0.001868 * np.cos(gam) - 0.032077 * np.sin(gam) - 0.014615 * np.cos(2 * gam) - 0.040849 * np.sin(2 * gam))
    dec = (0.006918 - 0.399912 * np.cos(gam) + 0.070257 * np.sin(gam) - 0.006758 * np.cos(2 * gam) + 0.000907 * np.sin(2 * gam)
           - 0.002697 * np.cos(3 * gam) + 0.00148 * np.sin(3 * gam))
    tst = hr * 60 + eot + 4 * lon                      # true solar time, minutes
    ha = np.radians(tst / 4 - 180)
    phi = np.radians(lat)
    cosz = np.clip(np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.cos(ha), -1, 1)
    zen = np.arccos(cosz)
    az = np.degrees(np.arctan2(np.sin(ha), np.cos(ha) * np.sin(phi) - np.tan(dec) * np.cos(phi))) + 180
    return np.degrees(zen), az % 360


def best_time_shift(df, lat, lon, shifts=(-60, -30, 0, 30, 60)):
    """Minutes to add to the POWER timestamp so that GHI best matches cos(zenith) (checks the hour convention)."""
    sub = df.loc[df.index.year == df.index.year.max()]
    out = {}
    for m in shifts:
        z, _ = solar_position(sub.index + pd.Timedelta(minutes=m), lat, lon)
        out[m] = np.corrcoef(np.clip(np.cos(np.radians(z)), 0, None), sub.ALLSKY_SFC_SW_DWN.fillna(0))[0, 1]
    return max(out, key=out.get), out


# ---------------------------------------------------------------- PV (isotropic sky, NOCT cell temperature, PVWatts-type DC)
def pv_output_per_kwp(df, lat, lon, tilt=None, az=None, albedo=0.25, gamma=-0.0037, noct=45.0, losses=0.14,
                      inv_eff=0.96, shift_min=30):
    """AC-side PV output (kW per kWp) each hour. Tilt defaults to |lat|, azimuth to the equator. Hourly irradiance is
    treated as the mean over the hour, evaluated at timestamp + shift_min."""
    tilt = abs(lat) if tilt is None else tilt
    az = (180.0 if lat >= 0 else 0.0) if az is None else az
    zen, saz = solar_position(df.index + pd.Timedelta(minutes=shift_min), lat, lon)
    z, b, sa, pa = map(np.radians, (zen, tilt, saz, az))
    cos_aoi = np.cos(z) * np.cos(b) + np.sin(z) * np.sin(b) * np.cos(sa - pa)
    ghi, dni, dhi = (df[c].fillna(0).clip(lower=0).values for c in ("ALLSKY_SFC_SW_DWN", "ALLSKY_SFC_SW_DNI", "ALLSKY_SFC_SW_DIFF"))
    up = zen < 90
    poa = (dni * np.clip(cos_aoi, 0, None) * up + dhi * (1 + np.cos(b)) / 2 + ghi * albedo * (1 - np.cos(b)) / 2)
    tcell = df["T2M"].values + poa / 800.0 * (noct - 20.0)
    p = poa / 1000.0 * (1 + gamma * (tcell - 25.0)) * (1 - losses) * inv_eff
    return pd.Series(np.clip(p, 0, None), index=df.index, name="pv_kw_per_kwp"), pd.Series(poa, index=df.index)


# ---------------------------------------------------------------- wind (shear from 10/50 m, density correction, generic curve)
def wind_output_per_kw(df, hub=30.0, v_in=3.0, v_r=12.0, v_out=25.0, alpha_bounds=(0.05, 0.40), losses=0.10):
    """Output (kW per kW rated) of a generic small turbine. Hourly shear exponent from the 10 m and 50 m POWER winds,
    hub-height speed by the power law, IEC density correction v*(rho/1.225)^(1/3), cubic curve between cut-in and
    rated speed, availability/electrical losses."""
    v10, v50 = df.WS10M.clip(lower=0.1).values, df.WS50M.clip(lower=0.1).values
    alpha = np.clip(np.log(v50 / v10) / np.log(50 / 10), *alpha_bounds)
    v = v50 * (hub / 50.0) ** alpha
    rho = df.PS.values * 1000 / (287.05 * (df.T2M.values + 273.15))
    v = v * (rho / 1.225) ** (1 / 3)
    p = np.where((v >= v_in) & (v < v_r), (v ** 3 - v_in ** 3) / (v_r ** 3 - v_in ** 3), 0.0)
    p = np.where((v >= v_r) & (v < v_out), 1.0, p) * (1 - losses)
    return pd.Series(p, index=df.index, name="wind_kw_per_kw"), pd.Series(v, index=df.index), pd.Series(alpha, index=df.index)


# ---------------------------------------------------------------- FAO-56 maize demand for sizing (single Kc)
def maize_kc(n_days, stages=(25, 40, 45, 30), kc=(0.30, 1.20, 0.35)):
    ini, dev, mid, late = stages
    k = np.empty(n_days)
    for d in range(n_days):
        if d < ini:
            k[d] = kc[0]
        elif d < ini + dev:
            k[d] = kc[0] + (kc[1] - kc[0]) * (d - ini) / dev
        elif d < ini + dev + mid:
            k[d] = kc[1]
        elif d < sum(stages):
            k[d] = kc[1] + (kc[2] - kc[1]) * (d - ini - dev - mid) / late
        else:
            k[d] = 0.0
    return k


def irrigation_demand(daily_et0, daily_rain, sow_mmdd, area_ha=5.0, eff=0.90, stages=(25, 40, 45, 30)):
    """Daily gross irrigation demand (m3) for maize sown each year on sow_mmdd: (Kc*ET0 - Peff)+ / efficiency.
    Peff = 0.8*(P - 5 mm)+ (simple daily effective-rain rule)."""
    out = pd.Series(0.0, index=daily_et0.index)
    L = sum(stages)
    for y in sorted(set(daily_et0.index.year)):
        s = pd.Timestamp(f"{y}-{sow_mmdd}")
        idx = pd.date_range(s, periods=L, freq="D")
        idx = idx[idx.isin(daily_et0.index)]
        if len(idx) == 0:
            continue
        k = maize_kc(L, stages)[:len(idx)]
        peff = 0.8 * np.clip(daily_rain.loc[idx].values - 5.0, 0, None)
        nir = np.clip(k * daily_et0.loc[idx].values - peff, 0, None)
        out.loc[idx] = nir / eff * 10.0 * area_ha           # mm over area_ha -> m3
    return out


# ---------------------------------------------------------------- year-round farm loads (appliance-based, stochastic, seeded)
FARM_LOAD = {   # ASSUMED appliance list (author query); RAMP-style: rated power, daily use window (local hours), daily minutes
    "lights":   dict(kw=0.06, window=(18, 23), minutes=240),
    "tv":       dict(kw=0.08, window=(19, 23), minutes=150),
    "workshop": dict(kw=2.00, window=(9, 17), minutes=120),
    "fridge_kw": 0.15, "fridge_duty": 0.40,
    "fans_kw": 0.15, "fans_tmax_c": 30.0,                      # fans run 11:00-18:00 local on days with Tmax above this
    "cold_ua_kw_per_k": 0.12, "cold_set_c": 2.0, "cold_cop": 2.0, "cold_base_kw": 0.30,
}


def farm_load(t2m_utc, lon, seed=0, cfg=FARM_LOAD):
    """Hourly year-round farm load (kW) from hourly air temperature (UTC index): household appliances in randomised daily
    windows (seeded), fridge duty cycle, fans on hot days, and a cold room whose compressor power follows
    UA*(T - Tset)/COP + base. Local time = UTC + lon/15 h (solar time, rounded)."""
    rng = np.random.default_rng(seed)
    idx = t2m_utc.index
    loc_hr = ((idx.hour + int(round(lon / 15))) % 24).values
    local_day = (idx + pd.Timedelta(hours=int(round(lon / 15)))).normalize()
    days, d_idx = np.unique(local_day.values, return_inverse=True)
    p = np.full(len(idx), cfg["fridge_kw"] * cfg["fridge_duty"])
    for name in ("lights", "tv", "workshop"):
        a = cfg[name]
        w0, w1 = a["window"]
        hours = max(1, int(round(a["minutes"] / 60)))
        starts = rng.integers(w0, max(w0 + 1, w1 - hours + 1), size=len(days))         # random start hour each day
        on = (loc_hr >= starts[d_idx]) & (loc_hr < starts[d_idx] + hours)
        p += on * a["kw"] * rng.uniform(0.8, 1.2, size=len(days))[d_idx]
    t = t2m_utc.values
    tmax_day = pd.Series(t).groupby(d_idx).transform("max").values
    p += ((tmax_day > cfg["fans_tmax_c"]) & (loc_hr >= 11) & (loc_hr < 18)) * cfg["fans_kw"]
    p += np.clip(t - cfg["cold_set_c"], 0, None) * cfg["cold_ua_kw_per_k"] / cfg["cold_cop"] + cfg["cold_base_kw"]
    return pd.Series(p, index=idx, name="farm_load_kw")


# ---------------------------------------------------------------- hourly system simulation (rule-based baseline dispatch)
def simulate(pv_kw, wind_kw, demand_m3_day, head_m, pv_kwp, wind_kwr, batt_kwh, tank_m3, pump_kw, load_kw=None,
             eta_pump=0.55, soc_min=0.10, soc_max=0.90, eta_ch=0.96, eta_dis=0.96, c_rate=0.5, pump_min_frac=0.3,
             tank0=0.5, pump_batt_reserve=0.5):
    """Rule-based dispatch. Each hour: renewables serve the farm load first (critical; battery covers any shortfall),
    then the pump (if the tank is not full), then charge the battery, then are curtailed. The battery may top up the
    pump to its minimum power only while the tank is below half or the day's demand is outstanding, and only above
    pump_batt_reserve of capacity (night reserve for the farm load). Irrigation is drawn from the tank at the first hour
    of each UTC day and from water pumped later that day. Returns hourly frame and summary dict.
    pv_kw / wind_kw: per-unit output series (hourly, aligned); demand: daily series (m3); load_kw: hourly kW or None."""
    n = len(pv_kw)
    gen = (pv_kw.values * pv_kwp + wind_kw.values * wind_kwr).tolist()
    load = (load_kw.reindex(pv_kw.index).fillna(0.0).values if load_kw is not None else np.zeros(n)).tolist()
    reserve = (soc_min + pump_batt_reserve * (soc_max - soc_min)) * batt_kwh
    unmet_e = 0.0
    kwh_per_m3 = RHO_W * G * head_m / eta_pump / 3.6e6
    days = pv_kw.index.normalize().tz_localize(None)
    uniq, day_idx = np.unique(days.values, return_inverse=True)
    dem_arr = demand_m3_day.reindex(pd.DatetimeIndex(uniq)).fillna(0.0).values.tolist()
    new_day = np.r_[True, day_idx[1:] != day_idx[:-1]].tolist()
    day_idx = day_idx.tolist()
    soc = soc_min * batt_kwh + 0.5 * (soc_max - soc_min) * batt_kwh
    tank = tank0 * tank_m3
    p_ch_max = c_rate * batt_kwh
    rec = np.zeros((n, 8))
    delivered = unmet = 0.0
    owed = 0.0
    for i in range(n):
        if new_day[i]:                                       # new day: settle yesterday, draw today's irrigation
            unmet += owed
            owed = dem_arr[day_idx[i]]
            take = min(owed, tank); tank -= take; owed -= take; delivered += take
        g = gen[i]
        L = load[i]
        served = min(g, L); g -= served; short = L - served   # farm load first
        if short > 0:
            dis = min(short, max(soc - soc_min * batt_kwh, 0.0) * eta_dis, p_ch_max)
            soc -= dis / eta_dis; short -= dis; served += dis
        unmet_e += short
        room = max(tank_m3 - tank, 0.0)
        p_pump = 0.0
        if room > 1e-9 and pump_kw > 0:
            p_need = min(pump_kw, room * kwh_per_m3)
            p_pump = min(g, p_need)
            if p_pump < pump_min_frac * pump_kw:            # VFD minimum: try to top up from battery when water is needed
                want = (tank < 0.5 * tank_m3) or owed > 0
                top = min(pump_min_frac * pump_kw - p_pump, max(soc - reserve, 0) * eta_dis, p_ch_max) if want else 0.0
                if p_pump + top >= pump_min_frac * pump_kw - 1e-9:
                    soc -= top / eta_dis; p_pump += top
                else:
                    p_pump = 0.0
        surplus = g - min(p_pump, g)
        ch = min(surplus, p_ch_max, max(soc_max * batt_kwh - soc, 0) / eta_ch)
        soc += ch * eta_ch
        curtail = surplus - ch
        water = p_pump / kwh_per_m3
        tank += water
        if owed > 0:                                         # water arriving later in the day still serves the demand
            take = min(owed, tank); tank -= take; owed -= take; delivered += take
        rec[i] = (gen[i], p_pump, ch, curtail, soc, tank, served, L)
    if owed > 0:
        unmet += owed
    h = pd.DataFrame(rec, index=pv_kw.index,
                     columns=["gen_kw", "pump_kw", "charge_kw", "curtail_kw", "soc_kwh", "tank_m3", "load_served_kw", "load_kw"])
    total_dem = float(sum(dem_arr)); total_load = float(sum(load))
    summ = dict(demand_m3=total_dem, delivered_m3=delivered, unmet_m3=unmet,
                wlpsp=unmet / total_dem if total_dem > 0 else 0.0, gen_kwh=h.gen_kw.sum(), pump_kwh=h.pump_kw.sum(),
                load_kwh=total_load, load_unmet_kwh=unmet_e, elpsp=unmet_e / total_load if total_load > 0 else 0.0,
                curtail_kwh=h.curtail_kw.sum(), batt_throughput_kwh=h.charge_kw.sum(), kwh_per_m3=kwh_per_m3)
    return h, summ


# ---------------------------------------------------------------- economics
COSTS = {   # base case, USD (2022-2025 dollar years, not re-indexed); sources and status per item in COST_SOURCES
    "pv_per_kwp": 691, "wind_per_kw": 6680, "batt_per_kwh": 350, "tank_per_m3": 100, "pump_per_kw": 900,
    "om_frac": {"pv": 19 / 691, "wind": 39 / 6680, "batt": 0.025, "tank": 0.01, "pump": 0.03},
    "life": {"pv": 25, "wind": 20, "batt": 15, "tank": 25, "pump": 10},
    "discount": 0.075, "project_years": 25,
    "diesel_usd_per_l": 1.00, "genset_kwh_per_l": 3.0, "genset_usd_per_kw": 400, "co2_kg_per_l": 2.68,
}

# 2024 average retail diesel (pump price incl. taxes) converted at 2024 average exchange rates; see COST_SOURCES
DIESEL_USD_PER_L = {"bushland_tx": 0.99, "jodhpur_in": 1.07, "souss_ma": 1.26, "azraq_jo": 1.00}

# capital-cost bounds for the sensitivity analysis (O&M scales with capital through om_frac)
COST_SCENARIOS = {
    "base": {},
    "capex_low": {"pv_per_kwp": 525, "wind_per_kw": 3775, "batt_per_kwh": 192},
    "capex_high": {"pv_per_kwp": 1780, "wind_per_kw": 9630, "batt_per_kwh": 466},
    "discount_4%": {"discount": 0.04},
    "discount_12%": {"discount": 0.12},
    "battery_life_10y": {"life": {**COSTS["life"], "batt": 10}},
    "diesel_-30%": {"diesel_scale": 0.7},
    "diesel_+30%": {"diesel_scale": 1.3},
    "tank_low": {"tank_per_m3": 85},
    "tank_high": {"tank_per_m3": 170},
    "tank_elevated_steel": {"tank_per_m3": 620, "tank_fixed_usd": 5200, "life": {**COSTS["life"], "tank": 20}},
    "pump_low": {"pump_per_kw": 125},
    "pump_high": {"pump_per_kw": 1180},
}

COST_SOURCES = {   # value basis and verification status, used for the manuscript cost table
    "pv_per_kwp": "691: IRENA RPGC in 2024 (2025), global weighted-average total installed cost of utility-scale PV, 2024 USD; "
                  "low 525: same report, India; high 1780: NREL ATB 2024b commercial PV (200 kWdc), 2023",
    "wind_per_kw": "6680: PNNL Distributed Wind Energy Technology Data Update 2025 ed., average installed cost of new small wind "
                   "(<=100 kW) projects in 2024, n=30; low 3775: PNNL 2026 ed., midsize/large turbines 2016-2022 average; "
                   "high 9630: PNNL 2026 ed., small wind 2025, n=5",
    "batt_per_kwh": "350: assumed, inside the sourced range; low 192: IRENA RPGC in 2024, utility-scale BESS 2024; "
                    "high 466: NREL ATB 2024b residential battery pack 283 + inverter 183 USD/kWh (2022 USD, hardware only)",
    "om_frac": "PV 19 USD/kW-yr: NREL ATB 2024b commercial PV (2023); wind 39 USD/kW-yr: NREL ATB 2024b distributed wind; "
               "battery 2.5%/yr of capital: NREL ATB 2024b residential/commercial battery; tank 1%, pump 3%: assumed",
    "life": "PV 25 yr: IRENA RPGC Table A1; wind 20 yr: IEC 61400-2 minimum design life for small turbines; battery 15 yr: "
            "NREL ATB 2024b; tank 25 yr, pump 10 yr: assumed",
    "discount": "7.5% real: IRENA RPGC Table A1 rest-of-world WACC since 2020; sensitivity 4% (Asia PV) and 12% (Africa PV): "
                "IRENA RPGC in 2024, Fig. 1.13",
    "tank_per_m3": "100: inside 85-170 USD/m3 for ground-level steel-frame tank kits of 95-45 m3 (UNICEF Supply Catalogue, "
                   "indicative prices); sensitivity low 85, high 170, and an elevated steel tank 620 USD/m3 + 5,200 USD, "
                   "life 20 yr (Soenen et al. 2021, Energies 14:2483, Table 2)",
    "pump_per_kw": "900: installed submersible pump with drive, inside the range from about 125 USD/kW (pump only, derived from "
                   "MNRE benchmark costs 2019-20: standalone solar pump minus pump solarisation) to about 1,180 USD/kW "
                   "(US NRCS FY2024 practice 533 electric pumping plant, payment rate at 75 % of cost); sensitivity 125 / 1180",
    "diesel_usd_per_l": "USA 0.99: EIA on-highway diesel 2024 average 3.76 USD/gal (FRED GASDESA); India 1.07: 89.9 INR/L, "
                        "PPAC Delhi daily RSP 2024 (mean 88.03) with Delhi taxes removed and Rajasthan VAT 17.30% + 1.75 INR/L "
                        "cess applied, at 83.7 INR/USD; Morocco 1.26: 12.51 MAD/L, mean of 24 fortnightly national average "
                        "pump prices TTC 2024 (Conseil de la Concurrence quarterly gasoil reportings) at 9.95 MAD/USD; "
                        "Jordan 1.00: 705.8 fils/L, mean of the 12 monthly pricing-committee prices 2024 (MEMR) at 1.41 USD/JOD",
    "genset_usd_per_kw": "400: within 260-510 USD/kVA (UNICEF supply catalogue, 20-30 kVA sets) and about 360 USD/kW "
                         "(World Bank mini-grid planning example)",
    "genset_kwh_per_l": "3.0: assumed full-load efficiency at all loads (favours diesel)",
    "co2_kg_per_l": "2.68: IPCC 2006 default diesel emission factor (74.1 t CO2/TJ) at about 36 MJ/L",
}


def site_costs(site_id, scenario="base"):
    """Base costs with the site's diesel price and one sensitivity scenario applied."""
    ch = dict(COST_SCENARIOS[scenario])
    scale = ch.pop("diesel_scale", 1.0)
    return {**COSTS, "diesel_usd_per_l": DIESEL_USD_PER_L[site_id] * scale, **ch}


def crf(i, n):
    return i * (1 + i) ** n / ((1 + i) ** n - 1)


def diesel_reference(kwh_per_year, pump_kw, peak_load_kw=0.0, c=COSTS):
    """NPC and annual CO2 of a diesel genset supplying the same energy (pumping + farm load). Constant fuel efficiency
    (genset_kwh_per_l) is assumed, which favours diesel at part load, so renewable savings are not overstated.
    Genset rated 1.25 x (pump + peak farm load), life 10 yr; pump capital included."""
    i, N = c["discount"], c["project_years"]
    litres = kwh_per_year / c["genset_kwh_per_l"]
    ann = (1 - (1 + i) ** -N) / i
    cap = (pump_kw + peak_load_kw) * 1.25 * c["genset_usd_per_kw"] + pump_kw * c["pump_per_kw"]
    npc_d = cap + sum(cap / (1 + i) ** t for t in range(10, N, 10)) + litres * c["diesel_usd_per_l"] * ann + cap * 0.05 * ann
    return dict(npc=npc_d, litres_per_year=litres, co2_t_per_year=litres * c["co2_kg_per_l"] / 1000)


GRID_DEFAULT = {"pv": [0, 1, 1.5, 2, 3, 4], "wind": [0, 0.5, 1, 2], "batt": [0, 1, 2, 4], "tank": [0.03, 0.0625, 0.125, 0.25, 0.5]}


def size_system(pv, wind, demand, head_m, train, test, pump_kw, peak_m3_day, load_kw=None, grid=None,
                wlpsp_max=0.05, elpsp_max=0.02, c=COSTS):
    """Grid search over PV, wind, battery and tank sizes (multiples of pump power / peak daily demand).
    Feasible = water loss-of-supply probability <= wlpsp_max AND farm-load energy loss-of-supply probability
    <= elpsp_max on the training years. Returns all configurations and the least-NPC design of any mix ('least_cost'),
    and the least-NPC 'hybrid' (PV and wind both > 0), 'pv_only' and 'wind_only' designs, each re-simulated on the test
    years (out-of-sample reliability)."""
    grid = grid or GRID_DEFAULT
    tr_years = len(set(pv.loc[train].index.year))
    ld_tr = load_kw.loc[train] if load_kw is not None else None
    peak_load = float(load_kw.max()) if load_kw is not None else 0.0
    rows = []
    for gp in grid["pv"]:
        for gw in grid["wind"]:
            if gp == 0 and gw == 0:
                continue
            for gb in grid["batt"]:
                for gt in grid["tank"]:
                    cfg = dict(pv_kwp=gp * pump_kw, wind_kwr=gw * pump_kw, batt_kwh=gb * pump_kw, tank_m3=gt * peak_m3_day)
                    _, s = simulate(pv.loc[train], wind.loc[train], demand.loc[train], head_m, pump_kw=pump_kw, load_kw=ld_tr, **cfg)
                    cost = npc(**cfg, pump_kw=pump_kw, c=c)
                    ann = cost * crf(c["discount"], c["project_years"])
                    used = s["pump_kwh"] + s["load_kwh"] - s["load_unmet_kwh"]
                    rows.append({**cfg, "npc_usd": cost, "wlpsp_train": s["wlpsp"], "elpsp_train": s["elpsp"],
                                 "curtail_frac": s["curtail_kwh"] / s["gen_kwh"], "lcoe_usd_kwh": ann / (used / tr_years),
                                 "pump_kwh_yr": s["pump_kwh"] / tr_years, "load_kwh_yr": s["load_kwh"] / tr_years,
                                 "batt_cycles_yr": (s["batt_throughput_kwh"] / cfg["batt_kwh"] / tr_years) if cfg["batt_kwh"] > 0 else 0.0})
    allc = pd.DataFrame(rows)
    feas = allc[(allc.wlpsp_train <= wlpsp_max) & (allc.elpsp_train <= elpsp_max)]
    pick = {}
    for name, sub in (("least_cost", feas), ("hybrid", feas[(feas.pv_kwp > 0) & (feas.wind_kwr > 0)]),
                      ("pv_only", feas[feas.wind_kwr == 0]), ("wind_only", feas[feas.pv_kwp == 0])):
        if len(sub):
            best = sub.sort_values("npc_usd").iloc[0].to_dict()
            _, st = simulate(pv.loc[test], wind.loc[test], demand.loc[test], head_m, pump_kw=pump_kw,
                             load_kw=load_kw.loc[test] if load_kw is not None else None,
                             **{k: best[k] for k in ("pv_kwp", "wind_kwr", "batt_kwh", "tank_m3")})
            best["wlpsp_test"], best["elpsp_test"] = st["wlpsp"], st["elpsp"]
            best["diesel"] = diesel_reference(best["pump_kwh_yr"] + best["load_kwh_yr"], pump_kw, peak_load, c)
            pick[name] = best
    return allc, pick


def npc(pv_kwp, wind_kwr, batt_kwh, tank_m3, pump_kw, c=COSTS):
    """Net present cost over the project life: capital + replacements (salvage ignored) + O&M."""
    i, N = c["discount"], c["project_years"]
    caps = {"pv": pv_kwp * c["pv_per_kwp"], "wind": wind_kwr * c["wind_per_kw"], "batt": batt_kwh * c["batt_per_kwh"],
            "tank": tank_m3 * c["tank_per_m3"] + (c.get("tank_fixed_usd", 0.0) if tank_m3 > 0 else 0.0),
            "pump": pump_kw * c["pump_per_kw"]}
    total = 0.0
    pv_annuity = (1 - (1 + i) ** -N) / i
    for k, cap in caps.items():
        L = c["life"][k]
        total += cap + sum(cap / (1 + i) ** t for t in range(L, N, L)) + cap * c["om_frac"][k] * pv_annuity
    return total

In [ ]:
%%writefile {SRC}/et0.py
"""FAO-56 Penman-Monteith daily reference evapotranspiration (Allen et al. 1998, eqs. 7-8, 11-13, 17, 21-25, 28-39, 6).

Inputs per day: Tmax, Tmin (deg C), mean RH (%), u2 wind at 2 m (m/s), Rs (MJ m-2 d-1), pressure (kPa) or elevation (m),
latitude (deg). Missing pressure is estimated from elevation (eq. 7).
"""
import numpy as np
import pandas as pd


def es(t):
    return 0.6108 * np.exp(17.27 * t / (t + 237.3))


def et0_fao56(date, tmax, tmin, rh_mean, u2, rs, lat_deg, elev_m, p_kpa=None):
    date = pd.to_datetime(pd.Series(date)).reset_index(drop=True)
    tmax, tmin, rh, u2, rs = (np.asarray(x, float) for x in (tmax, tmin, rh_mean, u2, rs))
    tmean = (tmax + tmin) / 2
    p = np.asarray(p_kpa, float) if p_kpa is not None else np.full_like(tmean, 101.3 * ((293 - 0.0065 * elev_m) / 293) ** 5.26)
    gamma = 0.000665 * p
    delta = 4098 * es(tmean) / (tmean + 237.3) ** 2
    e_s = (es(tmax) + es(tmin)) / 2
    e_a = rh / 100 * e_s                                   # eq. 19 (mean RH)
    j = date.dt.dayofyear.values
    phi = np.radians(lat_deg)
    dr = 1 + 0.033 * np.cos(2 * np.pi * j / 365)
    dec = 0.409 * np.sin(2 * np.pi * j / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(dec), -1, 1))
    ra = 24 * 60 / np.pi * 0.0820 * dr * (ws * np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.sin(ws))
    rso = (0.75 + 2e-5 * elev_m) * ra
    rns = (1 - 0.23) * rs
    rel = np.clip(rs / rso, 0.3, 1.0)
    rnl = 4.903e-9 * ((tmax + 273.16) ** 4 + (tmin + 273.16) ** 4) / 2 * (0.34 - 0.14 * np.sqrt(e_a)) * (1.35 * rel - 0.35)
    rn = rns - rnl
    et0 = (0.408 * delta * rn + gamma * 900 / (tmean + 273) * u2 * (e_s - e_a)) / (delta + gamma * (1 + 0.34 * u2))
    return np.maximum(et0, 0.0)

In [ ]:
%%writefile {SRC}/sites.json
{
  "bushland_tx": {"name": "Bushland, Texas, USA", "lat": 35.186, "lon": -102.094, "role": "crop-water validation (USDA-ARS weighing lysimeters) and case study", "region": "Southern High Plains, Ogallala aquifer"},
  "jodhpur_in": {"name": "Jodhpur, Rajasthan, India", "lat": 26.25, "lon": 73.02, "role": "case study", "region": "Thar desert margin"},
  "souss_ma": {"name": "Taroudant, Souss-Massa, Morocco", "lat": 30.47, "lon": -8.88, "role": "case study", "region": "Souss-Massa basin, solar-pumped groundwater irrigation"},
  "azraq_jo": {"name": "Azraq, Jordan", "lat": 31.83, "lon": 36.82, "role": "case study (held-out site candidate)", "region": "Azraq basin, over-abstracted aquifer"}
}

In [ ]:
import importlib, energy as en, et0 as et0m
importlib.reload(en); importlib.reload(et0m)
SITES = json.load(open(f'{SRC}/sites.json', encoding='utf-8'))
# ASSUMPTIONS for sizing (author queries): maize sowing date per site, pumping head, farm area
SOW = {'bushland_tx': '05-15', 'jodhpur_in': '07-01', 'souss_ma': '03-15', 'azraq_jo': '04-01'}
HEAD_M = 60.0; AREA_HA = 5.0; PUMP_HOURS = 8.0; LOAD_SEED = 1
TRAIN, TEST = slice('2001-01-01', '2016-12-31'), slice('2020-01-01', '2024-12-31')

## 2. Download NASA POWER hourly (2001–2024) and daily data (skips files already on Drive)

In [ ]:
HOURLY = 'ALLSKY_SFC_SW_DWN,ALLSKY_SFC_SW_DNI,ALLSKY_SFC_SW_DIFF,T2M,RH2M,PRECTOTCORR,WS10M,WS50M,PS'
DAILY = 'ALLSKY_SFC_SW_DWN,T2M_MAX,T2M_MIN,T2M,RH2M,WS2M,PRECTOTCORR,PS'
def fetch(url, tries=5):
    for k in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=300) as r: return r.read()
        except Exception as e:
            print('  retry', k + 1, e); time.sleep(10 * (k + 1))
    raise RuntimeError(url)
def save(path, payload):
    json.loads(payload)
    with gzip.open(path, 'wb') as f: f.write(payload)
for sid, s in SITES.items():
    d = f'{RAWP}/{sid}'; os.makedirs(d, exist_ok=True)
    f = f'{d}/{sid}_daily_2001_2024.json.gz'
    if not os.path.exists(f):
        q = dict(parameters=DAILY, community='AG', longitude=s['lon'], latitude=s['lat'], start='20010101', end='20241231', format='JSON')
        save(f, fetch('https://power.larc.nasa.gov/api/temporal/daily/point?' + urllib.parse.urlencode(q)))
    for y in range(2001, 2025):
        f = f'{d}/{sid}_hourly_{y}.json.gz'
        if os.path.exists(f): continue
        q = dict(parameters=HOURLY, community='RE', longitude=s['lon'], latitude=s['lat'], start=f'{y}0101', end=f'{y}1231',
                 format='JSON', **{'time-standard': 'UTC'})
        save(f, fetch('https://power.larc.nasa.gov/api/temporal/hourly/point?' + urllib.parse.urlencode(q))); time.sleep(1)
    print(sid, 'files:', len(os.listdir(d)))

## 3. Resource assessment, irrigation demand and processed tables (saved for notebooks 03+)

In [ ]:
R, summary = {}, []
for sid, s in SITES.items():
    h = en.load_power_hourly(f'{RAWP}/{sid}', sid)
    assert len(h) == 210384 and not h.isna().any().any(), (sid, len(h), int(h.isna().sum().sum()))
    shift, corr = en.best_time_shift(h, s['lat'], s['lon'])
    pv, poa = en.pv_output_per_kwp(h, s['lat'], s['lon'], shift_min=shift)
    wd, vhub, alpha = en.wind_output_per_kw(h)
    dly, elev = en.load_power_daily(f'{RAWP}/{sid}/{sid}_daily_2001_2024.json.gz')
    et0 = pd.Series(et0m.et0_fao56(dly.index, dly.T2M_MAX, dly.T2M_MIN, dly.RH2M, dly.WS2M, dly.ALLSKY_SFC_SW_DWN, s['lat'], elev, dly.PS), index=dly.index)
    dem = en.irrigation_demand(et0, dly.PRECTOTCORR, SOW[sid], area_ha=AREA_HA)
    load = en.farm_load(h.T2M, s['lon'], seed=LOAD_SEED)
    R[sid] = dict(pv=pv, wind=wd, dem=dem, load=load, shift=shift, elev=elev)
    hourly = pd.DataFrame({'pv_kw_per_kwp': pv, 'wind_kw_per_kw': wd, 'farm_load_kw': load, 'poa_w_m2': poa, 'v_hub': vhub,
                           'ghi': h.ALLSKY_SFC_SW_DWN, 't2m': h.T2M, 'rh2m': h.RH2M, 'ws50m': h.WS50M, 'rain_mm': h.PRECTOTCORR})
    hourly.to_parquet(f'{PROC}/{sid}_hourly.parquet')
    pd.DataFrame({'et0_mm': et0, 'rain_mm': dly.PRECTOTCORR, 'tmax': dly.T2M_MAX, 'tmin': dly.T2M_MIN, 'demand_m3': dem}).to_parquet(f'{PROC}/{sid}_daily.parquet')
    mpv, mwd, mdem = pv.resample('MS').mean(), wd.resample('MS').mean(), dem.resample('MS').sum()
    summary.append(dict(site=sid, name=s['name'], power_cell_elev_m=elev, timestamp_shift_min=shift,
                        pv_kwh_per_kwp_yr=pv.groupby(pv.index.year).sum().mean(), pv_yield_cv=pv.groupby(pv.index.year).sum().std() / pv.groupby(pv.index.year).sum().mean(),
                        poa_kwh_m2_yr=poa.groupby(poa.index.year).sum().mean() / 1000, wind_cf_30m=wd.mean(), mean_vhub=vhub.mean(), mean_shear=alpha.mean(),
                        et0_mm_yr=et0.groupby(et0.index.year).sum().mean(), demand_m3_season=dem.groupby(dem.index.year).sum().mean(),
                        peak_demand_m3_day=dem.loc[TRAIN].max(), farm_load_kwh_yr=load.groupby(load.index.year).sum().mean(),
                        farm_load_peak_kw=load.max(), r_hourly_pv_wind=np.corrcoef(pv, wd)[0, 1],
                        r_monthly_pv_wind=np.corrcoef(mpv, mwd)[0, 1]))
    print(sid, 'done')
S = pd.DataFrame(summary); S.to_csv(f'{OUT}/resource_summary.csv', index=False); S.round(3)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 3.6), sharey=False)
for a, (sid, r) in zip(ax, R.items()):
    m = lambda x: x.groupby(x.index.month).mean()
    a.plot(range(1, 13), m(r['pv']) * 24, label='PV, kWh/kWp/day'); a.plot(range(1, 13), m(r['wind']) * 24, label='Wind, kWh/kW/day')
    a.plot(range(1, 13), m(r['load']) * 24 / 10, ls=':', color='k', label='Farm load, kWh/day ÷10')
    a2 = a.twinx(); dm = r['dem'].groupby([r['dem'].index.year, r['dem'].index.month]).sum().groupby(level=1).mean()
    a2.bar(dm.index, dm.values, alpha=.25, color='tab:green', label='Irrigation demand'); a2.set_ylabel('m³/month', fontsize=8)
    a.set_title(SITES[sid]['name'], fontsize=9); a.set_xticks(range(1, 13)); a.legend(fontsize=7, loc='upper left')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_resource_demand.png', dpi=200); plt.show()

## 4. Cross-check of the PV irradiance model against pvlib (2020, same inputs)

In [ ]:
import pvlib
chk = []
for sid, s in SITES.items():
    h = en.load_power_hourly(f'{RAWP}/{sid}', sid).loc['2020']
    t = h.index + pd.Timedelta(minutes=R[sid]['shift'])
    sp = pvlib.solarposition.get_solarposition(t, s['lat'], s['lon'])
    tilt, az = abs(s['lat']), 180.0
    tot = pvlib.irradiance.get_total_irradiance(tilt, az, sp['apparent_zenith'].values, sp['azimuth'].values,
                                                h.ALLSKY_SFC_SW_DNI.values, h.ALLSKY_SFC_SW_DWN.values, h.ALLSKY_SFC_SW_DIFF.values,
                                                albedo=0.25, model='isotropic')
    poa_pvlib = pd.Series(np.nan_to_num(tot['poa_global']), index=h.index)
    _, poa_own = en.pv_output_per_kwp(h, s['lat'], s['lon'], shift_min=R[sid]['shift'])
    chk.append(dict(site=sid, poa_own_kwh_m2=poa_own.sum() / 1000, poa_pvlib_kwh_m2=poa_pvlib.sum() / 1000,
                    diff_pct=100 * (poa_own.sum() / poa_pvlib.sum() - 1), r2_hourly=np.corrcoef(poa_own, poa_pvlib)[0, 1] ** 2))
C = pd.DataFrame(chk); C.to_csv(f'{OUT}/pvlib_crosscheck.csv', index=False); C.round(3)

## 5. Sizing at each site (head 60 m; feasible if WLPSP ≤ 5% and ELPSP ≤ 2% on 2001–2016; reliability re-checked on 2020–2024)

In [ ]:
GRID = en.GRID_DEFAULT
picks, t0 = {}, time.time()
for sid, r in R.items():
    peak = r['dem'].loc[TRAIN].max()
    pump_kw = peak / PUMP_HOURS * en.RHO_W * en.G * HEAD_M / 0.55 / 3.6e6
    allc, pick = en.size_system(r['pv'], r['wind'], r['dem'], HEAD_M, TRAIN, TEST, pump_kw, peak, load_kw=r['load'], grid=GRID,
                                c=en.site_costs(sid))
    allc.to_csv(f'{OUT}/sizing_all_{sid}.csv', index=False)
    picks[sid] = {'pump_kw': pump_kw, 'peak_m3_day': peak, **pick}
    print(f'{sid}: {len(allc)} configs, {time.time() - t0:.0f}s, least-cost design:',
          {k: round(v, 1) for k, v in pick['least_cost'].items() if k in ('pv_kwp', 'wind_kwr', 'batt_kwh', 'tank_m3', 'npc_usd')})
json.dump(picks, open(f'{OUT}/sizing_picks.json', 'w'), indent=1, default=float)
rows = []
for sid, p in picks.items():
    for kind in ('least_cost', 'hybrid', 'pv_only', 'wind_only'):
        if kind in p:
            b = p[kind]
            rows.append(dict(site=sid, design=kind, pump_kw=p['pump_kw'], pv_kwp=b['pv_kwp'], wind_kw=b['wind_kwr'], batt_kwh=b['batt_kwh'],
                             tank_m3=b['tank_m3'], npc_usd=b['npc_usd'], lcoe_usd_kwh=b['lcoe_usd_kwh'], curtail_frac=b['curtail_frac'],
                             batt_cycles_yr=b['batt_cycles_yr'], wlpsp_train=b['wlpsp_train'], wlpsp_test=b['wlpsp_test'],
                             elpsp_train=b['elpsp_train'], elpsp_test=b['elpsp_test'],
                             diesel_npc_usd=b['diesel']['npc'], co2_avoided_t_yr=b['diesel']['co2_t_per_year']))
D = pd.DataFrame(rows); D.to_csv(f'{OUT}/sizing_summary.csv', index=False); D.round(3)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
for a, sid in zip(ax, R):
    allc = pd.read_csv(f'{OUT}/sizing_all_{sid}.csv')
    for lab, m, c in (('PV only', (allc.wind_kwr == 0), 'tab:orange'), ('Wind only', (allc.pv_kwp == 0), 'tab:blue'),
                      ('Hybrid', (allc.pv_kwp > 0) & (allc.wind_kwr > 0), 'tab:green')):
        a.scatter(allc.loc[m, ['wlpsp_train', 'elpsp_train']].max(axis=1) * 100, allc.loc[m, 'npc_usd'] / 1e3, s=8, alpha=.6, label=lab, color=c)
    a.axvline(5, color='k', lw=.6, ls='--'); a.set_xlabel('max(WLPSP, ELPSP) 2001–2016 (%)'); a.set_ylabel('NPC (k USD)')
    a.set_xlim(-1, 40); a.set_title(SITES[sid]['name'], fontsize=9); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_sizing_tradeoff.png', dpi=200); plt.show()

## 6. Sensitivity: sourced cost bounds, discount rate, battery life, diesel price (re-scored, same simulations) and pumping head (re-sized on a reduced grid; battery and PV reach 4x pump power because the farm load does not scale with head)

In [ ]:
sens = []
for sid in R:
    allc = pd.read_csv(f'{OUT}/sizing_all_{sid}.csv'); feas = allc[(allc.wlpsp_train <= 0.05) & (allc.elpsp_train <= 0.02)].copy(); pk = picks[sid]
    for name in en.COST_SCENARIOS:
        c = en.site_costs(sid, name)
        feas['npc_s'] = [en.npc(r.pv_kwp, r.wind_kwr, r.batt_kwh, r.tank_m3, pk['pump_kw'], c=c) for r in feas.itertuples()]
        b = feas.sort_values('npc_s').iloc[0]
        dz = en.diesel_reference(b.pump_kwh_yr + b.load_kwh_yr, pk['pump_kw'], float(R[sid]['load'].max()), c)
        sens.append(dict(site=sid, scenario=name, pv_kwp=b.pv_kwp, wind_kw=b.wind_kwr, batt_kwh=b.batt_kwh, tank_m3=b.tank_m3,
                         npc_usd=b.npc_s, diesel_npc_usd=dz['npc'], renewable_cheaper=b.npc_s < dz['npc']))
for sid, r in R.items():
    for head in (30.0, 90.0, 120.0):
        peak = r['dem'].loc[TRAIN].max(); pump_kw = peak / PUMP_HOURS * en.RHO_W * en.G * head / 0.55 / 3.6e6
        _, pk = en.size_system(r['pv'], r['wind'], r['dem'], head, TRAIN, TEST, pump_kw, peak, load_kw=r['load'],
                               grid={'pv': [1, 1.5, 2, 3, 4], 'wind': [0, 0.5, 1], 'batt': [0, 1, 2, 3, 4], 'tank': [0.03, 0.0625, 0.125, 0.25]},
                               c=en.site_costs(sid))
        b = pk.get('least_cost')
        if b:
            sens.append(dict(site=sid, scenario=f'head_{int(head)}m', pv_kwp=b['pv_kwp'], wind_kw=b['wind_kwr'], batt_kwh=b['batt_kwh'],
                             tank_m3=b['tank_m3'], npc_usd=b['npc_usd'], diesel_npc_usd=b['diesel']['npc'],
                             renewable_cheaper=b['npc_usd'] < b['diesel']['npc']))
SENS = pd.DataFrame(sens); SENS.to_csv(f'{OUT}/sensitivity.csv', index=False); SENS.round(1)

## 7. Export for checking (`export_nb02.zip`)

In [ ]:
info = {'run_utc': pd.Timestamp.utcnow().isoformat(), 'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__,
        'pvlib': pvlib.__version__, 'sow': SOW, 'head_m': HEAD_M, 'area_ha': AREA_HA, 'pump_hours': PUMP_HOURS, 'grid': GRID,
        'farm_load_assumed': en.FARM_LOAD, 'load_seed': LOAD_SEED, 'wlpsp_max': 0.05, 'elpsp_max': 0.02,
        'costs': en.COSTS, 'diesel_usd_per_l': en.DIESEL_USD_PER_L, 'cost_scenarios': en.COST_SCENARIOS,
        'cost_sources': en.COST_SOURCES, 'train': [TRAIN.start, TRAIN.stop], 'test': [TEST.start, TEST.stop]}
json.dump(info, open(f'{OUT}/run_info.json', 'w'), indent=1, default=str)
shutil.make_archive(f'{BASE}/outputs/export_nb02', 'zip', OUT)
files.download(f'{BASE}/outputs/export_nb02.zip')